<a href="https://colab.research.google.com/github/rouuuuuuu/PFA/blob/main/DeBERTa_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Installer les packages nécessaires
!pip install transformers datasets scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 18.0 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system 

In [ ]:

# Imports
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import DebertaV2Tokenizer, DebertaV2Model, get_scheduler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Charger CSV
df = pd.read_csv('/content/df_small_balanced_9000.csv')

# Vérifier les colonnes
print(df.columns)

# On travaille sur 'CommentText' et 'Sentiment'
text_column = 'CommentText'
label_column = 'Sentiment'

# Encodage des labels
le = LabelEncoder()
df['label'] = le.fit_transform(df[label_column])

# Diviser les données
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df[text_column].tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=42
)


Index(['CommentID', 'VideoID', 'VideoTitle', 'AuthorName', 'AuthorChannelID',
       'CommentText', 'Sentiment', 'Likes', 'Replies', 'PublishedAt',
       'CountryCode', 'CategoryID'],
      dtype='object')


In [ ]:
# Charger le tokenizer DeBERTa-v3
tokenizer = DebertaV2Tokenizer.from_pretrained('microsoft/deberta-v3-small')

# Dataset PyTorch
class CommentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(label, dtype=torch.long)
        }


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

In [ ]:
import torch
import torch.nn as nn
from transformers import DebertaV2Model

class DebertaClassifier(nn.Module):
    def __init__(self, num_labels):
        super(DebertaClassifier, self).__init__()
        self.deberta = DebertaV2Model.from_pretrained('microsoft/deberta-v3-small')
        self.classifier = nn.Linear(self.deberta.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.deberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0, :]  # CLS token
        return self.classifier(pooled_output)


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_dataset = CommentDataset(train_texts, train_labels, tokenizer)
test_dataset = CommentDataset(test_texts, test_labels, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

model = DebertaClassifier(num_labels=len(le.classes_)).to(device)

optimizer = AdamW(model.parameters(), lr=1e-5)
num_epochs = 50
num_training_steps = num_epochs * len(train_loader)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps
)

loss_fn = nn.CrossEntropyLoss()
for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()
        lr_scheduler.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {avg_loss:.4f}")


pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/286M [00:00<?, ?B/s]

Epoch 1/50, Loss: 0.6493
Epoch 2/50, Loss: 0.4215
Epoch 3/50, Loss: 0.3379
Epoch 4/50, Loss: 0.2544
Epoch 5/50, Loss: 0.1897
Epoch 6/50, Loss: 0.1366
Epoch 7/50, Loss: 0.0925
Epoch 8/50, Loss: 0.0737
Epoch 9/50, Loss: 0.0465
Epoch 10/50, Loss: 0.0356
Epoch 11/50, Loss: 0.0369
Epoch 12/50, Loss: 0.0239
Epoch 13/50, Loss: 0.0236
Epoch 14/50, Loss: 0.0231
Epoch 15/50, Loss: 0.0233
Epoch 16/50, Loss: 0.0186
Epoch 17/50, Loss: 0.0169
Epoch 18/50, Loss: 0.0194
Epoch 19/50, Loss: 0.0198
Epoch 20/50, Loss: 0.0086
Epoch 21/50, Loss: 0.0122
Epoch 22/50, Loss: 0.0162
Epoch 23/50, Loss: 0.0141
Epoch 24/50, Loss: 0.0064
Epoch 25/50, Loss: 0.0067
Epoch 26/50, Loss: 0.0062
Epoch 27/50, Loss: 0.0095
Epoch 28/50, Loss: 0.0092
Epoch 29/50, Loss: 0.0087
Epoch 30/50, Loss: 0.0051
Epoch 31/50, Loss: 0.0044
Epoch 32/50, Loss: 0.0062
Epoch 33/50, Loss: 0.0087
Epoch 34/50, Loss: 0.0057
Epoch 35/50, Loss: 0.0051
Epoch 36/50, Loss: 0.0050
Epoch 37/50, Loss: 0.0058
Epoch 38/50, Loss: 0.0044
Epoch 39/50, Loss: 0.

In [ ]:
model.eval()
all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask)
        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(probs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs[:, 1].cpu().numpy())  # probabilité positive pour AUC

# Calcul des métriques
accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, average='weighted')
recall = recall_score(all_labels, all_preds, average='weighted')
f1 = f1_score(all_labels, all_preds, average='weighted')
try:
    auc = roc_auc_score(all_labels, all_probs)
except ValueError:
    auc = 'N/A (non-binaire)'

print(f"TEST | Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}, AUC: {auc}")


TEST | Accuracy: 0.8300, Precision: 0.8309, Recall: 0.8300, F1: 0.8302, AUC: N/A (non-binaire)
